# Transformer embeddings for the Maintenance RAG — measured vs. the LSA baseline

Builds a second vector index using **sentence-transformers (`all-MiniLM-L6-v2`)** over the *same*
corpus and evaluates retrieval on the *same* labeled QA set, so the transformer embedder can be
compared head-to-head with the classical LSA baseline the repo ships with.

This runs in Colab because the model hub isn't reachable from the project's build sandbox (the LSA
baseline is fully offline; the transformer is the measured upgrade). Runtime → Run all. A CPU
runtime is fine — the corpus is small.


In [ ]:
!pip -q install sentence-transformers faiss-cpu 2>/dev/null
print('ready')

In [ ]:
# Clone the repo to reuse its corpus, QA set, chunking, and eval function.
REPO_URL = "https://github.com/SelenaG4/manufacturing-maintenance-rag.git"   # <-- edit if different
import sys, os, subprocess
if not os.path.isdir("manufacturing-maintenance-rag"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
%cd manufacturing-maintenance-rag
sys.path.insert(0, ".")

In [ ]:
import json, numpy as np, faiss
from sentence_transformers import SentenceTransformer
from app.corpus import load_chunks
from app.config import EVAL_KS
from scripts.evaluate_rag import load_qa, evaluate

chunks = load_chunks()
qa = load_qa()
print(len(chunks), "chunks,", len(qa), "eval questions")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = model.encode([c.text for c in chunks], normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(doc_vecs.shape[1]); index.add(doc_vecs)
chunk_ids = [c.chunk_id for c in chunks]

def transformer_search(q):
    qv = model.encode([q], normalize_embeddings=True).astype("float32")
    _, I = index.search(qv, max(EVAL_KS))
    return [chunk_ids[i] for i in I[0] if i >= 0]

In [ ]:
# Evaluate the transformer with the SAME metric function, and compare to the committed LSA numbers.
trans = evaluate(transformer_search, qa)
lsa = json.load(open("docs/retrieval_results_lsa.json"))

print("="*58)
print(f"{'Metric':<12}{'LSA baseline':>16}{'MiniLM':>12}{'Δ':>10}")
print("="*58)
for key in ["mrr"] + [f"hit@{k}" for k in EVAL_KS] + [f"recall@{k}" for k in EVAL_KS]:
    d = trans[key] - lsa[key]
    print(f"{key:<12}{lsa[key]:>16.3f}{trans[key]:>12.3f}{d:>+10.3f}")
print("="*58)
verdict = "beats" if trans["mrr"] > lsa["mrr"] else "does NOT beat"
print(f"On MRR, the transformer {verdict} the LSA baseline — measured, not assumed.")

json.dump({"lsa": lsa, "minilm": trans}, open("retrieval_comparison.json", "w"), indent=2)
print("\nSaved retrieval_comparison.json")

In [ ]:
# Download the comparison so it can go in the repo's README.
from google.colab import files
files.download("retrieval_comparison.json")